# Fraud Detection Case - Intern Data Scientist Brazil
**Author:** Davi de Souza Santos

This notebook develops a supervised fraud detection solution using labeled 2025 transactions and scores the unlabeled January 2026 transactions. The workflow follows the case requirements and keeps validation chronological.

## Notebook structure

1. [Executive Summary](#00-executive-summary)
2. [Setup and Data Loading](#01-setup-and-data-loading)
3. [Exploratory Data Analysis](#02-exploratory-data-analysis)
4. [Feature Engineering](#03-feature-engineering)
5. [Model Development and Temporal Validation](#04-model-development-and-temporal-validation)
6. [January 2026 Scoring and Deliverables](#05-january-2026-scoring-and-deliverables)

# 00. Summary

## 00.1 Goals

### 00.1.1 Business problem

The Fraud Prevention team of a financial institution has a consolidated process to review suspicious transactions. As the volume of operations grows, this process, which is largely manual, has become inefficient and hard to scale. This creates a double challenge:

- **Financial loss:** fraudulent transactions that are not detected in time.
- **Customer experience:** legitimate customers whose transactions are wrongly blocked.

The two errors have different costs. A missed fraud (false negative) is a direct loss, while a blocked legitimate transaction (false positive) hurts the customer relationship. Any solution has to balance both, and fraud is a rare event, so a model that simply labels everything as "not fraud" would look accurate while being useless.

The institution wants a Machine Learning solution that automates the identification of potentially fraudulent transactions, using a historical dataset in which part of the transactions were already labeled as fraud.

### 00.1.2 Defined Goals

**Main goal:** build a predictive model that classifies each transaction as *fraudulent* or *non-fraudulent*, and use it to classify the January 2026 transactions, which have no label.

**Specific goals:**

1. **Explore the data** (EDA) to understand what separates fraud from legitimate transactions, check data quality, and verify that January 2026 behaves like the 2025 history used for training.
2. **Engineer features** that capture the patterns found in the EDA, using only information available at the moment of the transaction.
3. **Train and compare models**, starting from a simple benchmark, and keep the simplest model that performs well and can be explained.
4. **Evaluate with metrics suited to rare events** (precision, recall, F1 and PR-AUC) on a temporal validation, where the model is trained on earlier periods and tested on later ones, as it would be used in practice.
5. **Choose and justify a decision threshold** in business terms, balancing undetected fraud against blocked legitimate customers.
6. **Deliver the results:** the January 2026 base with the classification and a presentation with the method, the results.

**Scope and limits:** the January 2026 base has no labels, so its classification cannot be measured directly. Performance is estimated on held-out periods of 2025, and the conclusions about January rely on the assumption that its behavior is similar to the training data, which is checked in the EDA.


## 00.2 Results
### 00.2.1 Conclusions and Recommendations
### 00.2.2 EDA
#### 00.2.2.1 Questions and their inferences
#### 00.2.2.2 What the data cannot answer
### 00.2.3 Feature Engineering
#### 00.2.3.1 Features chosen and why
#### 00.2.3.2 Validation Strategy
### 00.2.4 Model
#### 00.2.4.1 Machine Learning model chosen and why
#### 00.2.4.2 Results
#### 00.2.4.3 Threshold

## 00.3 Limitations

# 01. Setup and Data Loading

In [ ]:
from pathlib import Path
 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2
from sklearn.metrics import average_precision_score, mutual_info_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier, export_text

pd.set_option("display.max_columns", 50)

# Data paths
# 1. Training set
DATA_2025 = Path("data/transacoes_historico_2025.csv")
# 2. Scoring set
DATA_2026 = Path('data/transacoes_jan2026.csv')
# 3. Resulting figures path
FIG_DIR = Path("outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Target definition
TARGET = "flag_fraude"

def save_fig(name: str) -> None:
    """Save the current figure so it can be reused in the presentation."""
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
df = pd.read_csv(DATA_2025, parse_dates=["datahora_transacao"])
df_jan =pd.read_csv(DATA_2026, parse_dates=["datahora_transacao"])

print("2025 (train):", df.shape)
print("Jan 2026 (test):", df_jan.shape)
df.head()

# 02. Exploratory Data Analysis

Goals of this step:

1. Understand the structure and data quality.
2. Understand the target (`flag_fraude`) in the historic 2025 database.
3. Find which variables separate fraud from non-fraud.
4. Check which columns are redundant or useless.
5. Check wheter January 2026 (the testing set) looks like the training data.

## 02.1 Questions answered by EDA

#### Question 02.1.1: Is the fraud rate and the fraud pattern stable over the year?

#### Questionn 02.1.2: Which interactions matter beyond single variables?

#### Question 02.1.3: Which variables carry information, ranked quantitatively?

#### Question 02.1.4: How far can a very simple rule-based benchmark go?
> Show the results that the simple benchmark made

#### Question 02.1.5: What is the financial profile of fraud?

#### Question 02.1.6: Does customer history help?
> Features like velocity, new city, new merchant, repeat victims, help to detect a fraud?

#### Question 02.1.7: Are frauds clustered in time at the same merchant (bursts)?

#### Question 02.1.8: Does January 2026 look like 2025 history?

## 02.2 Structure and data quality
Basic quality checks

In [ ]:
print("Columns only in train :", set(df.columns) - set(df_jan.columns))
print("Columns only in Jan 26:", set(df_jan.columns) - set(df.columns))
print("\nMissing values (train):", int(df.isna().sum().sum()))
print("Duplicated rows (train):", int(df.duplicated().sum()))
print("Duplicated rows (Jan 26):", int(df_jan.duplicated().sum()))
print("\nTime coverage (train):", df.datahora_transacao.min(), "->", df.datahora_transacao.max())
print("Time coverage (Jan 26):", df_jan.datahora_transacao.min(), "->", df_jan.datahora_transacao.max())
df.dtypes


Sort sets by time

In [ ]:
df = df.sort_values("datahora_transacao").reset_index(drop=True)
df_jan = df_jan.sort_values("datahora_transacao").reset_index(drop=True)
 
for d in (df, df_jan):
    d["month"] = d.datahora_transacao.dt.to_period("M")
    d["hour"] = d.datahora_transacao.dt.hour
    d["dayofweek"] = d.datahora_transacao.dt.day_of_week
    d["day"] = d.datahora_transacao.dt.day
    d["date"] = d.datahora_transacao.dt.normalize()
 
print("2025:", df.shape, "|", df.datahora_transacao.min(), "->", df.datahora_transacao.max())
print("Jan 2026:", df_jan.shape)


## 02.3 Target: class balance

In [ ]:
counts = df[TARGET].value_counts()
rate = df[TARGET].mean()
print(counts.to_string())
print(f"\nFraud rate: {rate:.2%}  (about 1 fraud every {1 / rate:.0f} transactions)")
print("Accuracy of a model that always predicts 'not fraud':", f"{1 - rate:.2%}")


### 02.3.1 Fraud rate over time

Monthly overview

In [1]:
monthly = df.groupby("month").agg(
    transactions=(TARGET, "size"),
    frauds=(TARGET, "sum"),
    fraud_rate=(TARGET, "mean"),
    avg_amount=("valor_transacao", "mean"),
    new_device_share=("is_new_device", "mean"),
)
print(monthly.round(4).to_string())
 
ax = monthly["fraud_rate"].plot.bar(figsize=(9, 3.5), color="#3b6ea5")
ax.axhline(df[TARGET].mean(), color="red", ls="--", lw=1, label="overall rate")
ax.set_ylabel("fraud rate")
ax.set_title("Fraud rate by month (2025)")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.1%}")
ax.legend()
save_fig("monthly_fraud_rate")

Weekly trend and abnormal days

In [ ]:
weekly = df.set_index("datahora_transacao")[TARGET].resample("W").mean()
ax = weekly.plot(figsize=(10, 3.5), marker="o", ms=3)
ax.axhline(df[TARGET].mean(), color="red", ls="--", lw=1)
ax.set_title("Weekly fraud rate")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.1%}")
save_fig("weekly_fraud_rate")


Days whose fraud count is far from the overall rate (binomial z-score)

In [ ]:
p = df[TARGET].mean()
daily = df.groupby("date")[TARGET].agg(n="size", frauds="sum")
daily["z"] = (daily.frauds - daily.n * p) / np.sqrt(daily.n * p * (1 - p))
print("\nDays with |z| > 3 (unusual fraud counts):")
unusual = daily[daily.z.abs() > 3]
print(unusual.round(2).to_string() if len(unusual) else "none")
print("\nNumber of days analysed:", len(daily), "| expected by chance at |z|>3: ~", round(len(daily) * 0.0027, 1))

Day of month and day of week

In [ ]:
print("Fraud rate by day of month:")
print(df.groupby("day")[TARGET].mean().round(4).to_string())
print("\nFraud rate by day of week (0 = Monday):")
print(df.groupby("dayofweek")[TARGET].mean().round(4).to_string())

Special dates

In [ ]:
special = {
    "New Year's Day": ("2025-01-01", "2025-01-01"),
    "Carnival": ("2025-03-01", "2025-03-05"),
    "Black Friday": ("2025-11-28", "2025-11-28"),
    "Christmas": ("2025-12-24", "2025-12-25"),
    "New Year's Eve": ("2025-12-31", "2025-12-31"),
}
rows = []
for name, (a, b) in special.items():
    m = (df.date >= a) & (df.date <= b)
    if m.sum():
        rows.append({"event": name, "n": int(m.sum()), "fraud_rate": df.loc[m, TARGET].mean(),
                     "avg_amount": df.loc[m, "valor_transacao"].mean()})
rows.append({"event": "all other days", "n": int(len(df)), "fraud_rate": p, "avg_amount": df.valor_transacao.mean()})
print(pd.DataFrame(rows).round(4).to_string(index=False))


### 02.3.2 Are the key relationships stable month by month?
Hour x month heatmap

In [ ]:
hm = df.pivot_table(index="hour", columns="month", values=TARGET, aggfunc="mean")
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(hm.values, aspect="auto", cmap="Reds")
ax.set_yticks(range(len(hm.index)), hm.index)
ax.set_xticks(range(len(hm.columns)), [str(c) for c in hm.columns], rotation=45)
ax.set_title("Fraud rate by hour and month")
fig.colorbar(im, label="fraud rate")
save_fig("hour_by_month_heatmap")

### 02.3.3 Fraud rate by categorical variable

In [ ]:
def fraud_by(col: str, data: pd.DataFrame = df) -> pd.DataFrame:
    out = data.groupby(col)[TARGET].agg(n="size", frauds="sum", fraud_rate="mean")
    return out.sort_values("fraud_rate", ascending=False)
 
 
def plot_rate(col: str, data: pd.DataFrame = df, name: str | None = None) -> None:
    t = fraud_by(col, data)
    ax = t["fraud_rate"].plot.bar(figsize=(7, 3.5), color="#3b6ea5")
    ax.axhline(data[TARGET].mean(), color="red", ls="--", lw=1, label="overall rate")
    ax.set_ylabel("fraud rate")
    ax.set_title(f"Fraud rate by {col}")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.legend()
    save_fig(name or f"fraud_rate_{col}")

Categorical variables

In [ ]:
for c in ["tipo_estabelecimento", "nome_estabelecimento", "cidade", "device_type", "is_new_device", "turno"]:
    print(f"\n--- {c} ---")
    print(fraud_by(c).round(4).to_string())

Plots for the strongest categorical signals

In [ ]:
plot_rate("is_new_device")
plot_rate("tipo_estabelecimento")
plot_rate("cidade")
plot_rate("device_type")

### 02.3.4 Time of day

Fraud rate by hour

In [ ]:
df["hour"] = df.datahora_transacao.dt.hour
df["dayofweek"] = df.datahora_transacao.dt.day_of_week
 
hourly = df.groupby("hour")[TARGET].agg(n="size", frauds="sum", fraud_rate="mean")
print(hourly.round(4).to_string())
 
ax = hourly["fraud_rate"].plot.bar(figsize=(10, 3.5), color="#3b6ea5")
ax.set_title("Fraud rate by hour of day")
ax.set_ylabel("fraud rate")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
save_fig("fraud_rate_hour")
 
print("\nBy day of week (0 = Monday):")
print(df.groupby("dayofweek")[TARGET].mean().round(4).to_string())


Window of hours and new device by month

In [ ]:
def hour_window(h: int) -> str:
    if h in (22, 23):
        return "22-23h"
    if 12 <= h <= 14:
        return "12-14h"
    if h <= 8:
        return "00-08h"
    return "09-11h & 15-21h"
 
 
df["hour_window"] = df.hour.map(hour_window)
print("Fraud rate by hour window and month:")
print(df.pivot_table(index="hour_window", columns="month", values=TARGET, aggfunc="mean").round(4).to_string())
print("\nFraud rate by new device and month:")
print(df.pivot_table(index="is_new_device", columns="month", values=TARGET, aggfunc="mean").round(4).to_string())

### 02.3.5 Interactions

Hour x new device

In [ ]:
hd = df.pivot_table(index="hour", columns="is_new_device", values=TARGET, aggfunc="mean")
print(hd.round(4).to_string())
ax = hd.plot(figsize=(10, 3.5), marker="o", ms=3)
ax.set_title("Fraud rate by hour, split by new device")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
save_fig("hour_by_new_device")

Amount x Merchant type

In [ ]:
AMOUNT_EDGES = [0, 8, 12, 16, 20, 25, 31, 39, 52, 79, np.inf]  # taken from the sample deciles
df["amount_bin"] = pd.cut(df.valor_transacao, AMOUNT_EDGES)
print("Fraud rate by amount bin and merchant type:")
print(df.pivot_table(index="amount_bin", columns="tipo_estabelecimento", values=TARGET,
                     aggfunc="mean", observed=True).round(4).to_string())
print("\nFraud amount by merchant type:")
print(df[df[TARGET]].groupby("tipo_estabelecimento")["valor_transacao"].describe().round(2).to_string())

Hour window x merchant type

In [ ]:
print("Fraud rate by hour window and merchant type:")
print(df.pivot_table(index="hour_window", columns="tipo_estabelecimento", values=TARGET, aggfunc="mean").round(4).to_string())

> The pivot tables above suggests that the risky hours depend on the merchant type.

City x Merchant type

In [ ]:
print(df.pivot_table(index="cidade", columns="device_type", values=TARGET, aggfunc="mean").round(4).to_string())

New device x Day shift

In [ ]:
inter = df.pivot_table(index="is_new_device", columns="turno", values=TARGET, aggfunc="mean")
print(inter[["madrugada", "manha", "tarde", "noite"]].round(4).to_string())

### 02.3.6 Transaction amount
Amount distribution by class

In [ ]:
print(df.groupby(TARGET)["valor_transacao"].describe().round(2).to_string())
 
df["amount_decile"] = pd.qcut(df.valor_transacao, 10, duplicates="drop")
dec = df.groupby("amount_decile", observed=True)[TARGET].agg(n="size", fraud_rate="mean")
print("\nFraud rate by amount decile:")
print(dec.round(4).to_string())
 
ax = dec["fraud_rate"].reset_index(drop=True).plot(marker="o", figsize=(7, 3.5))
ax.set_xlabel("amount decile (0 = cheapest)")
ax.set_ylabel("fraud rate")
ax.set_title("Fraud rate by amount decile")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
save_fig("fraud_rate_amount_decile")

### 02.3.7 No signal variables
Variables with apparently no signal: `score_interno`, `temperature`

In [ ]:
df["score_bin"] = pd.cut(df.score_interno, bins=range(0, 1100, 100), include_lowest=True)
print("Fraud rate by score_interno bin:")
print(df.groupby("score_bin", observed=True)[TARGET].mean().round(4).to_string())
 
print("\ntemperatura_celsius by class:")
print(df.groupby(TARGET)["temperatura_celsius"].describe().round(2).to_string())
print("\nTemperature < 0 C:", int((df.temperatura_celsius < 0).sum()), "rows")


For the brazilian cities, having 637 logs with negative temperature is a bit strange

## 02.4 Redundant columns
Derived columns check

In [ ]:
print("valor_categoria vs amount thresholds:")
print(df.groupby("valor_categoria")["valor_transacao"].agg(["min", "max"]).to_string())
 
print("\ncidade -> cidade_regiao is one-to-one?",
      df.groupby("cidade")["cidade_regiao"].nunique().max() == 1)
 
print("\nturno vs hour ranges:")
print(df.groupby("turno")["hour"].agg(["min", "max"]).to_string())
 
print("\ncodigo_postal unique values:", df.codigo_postal.nunique(), "of", len(df), "rows")


## 02.5 Customer level

In [ ]:
per_client = df.groupby("nome_cliente").agg(n_tx=(TARGET, "size"), n_fraud=(TARGET, "sum"))
print("Customers:", len(per_client))
print("Transactions per customer:\n", per_client.n_tx.describe().round(2).to_string())
print("\nCustomers with >= 1 fraud:", int((per_client.n_fraud >= 1).sum()))
print("Customers with >= 2 frauds:", int((per_client.n_fraud >= 2).sum()))

## 02.6 Training history versus January 2026

Drift check on categorical shares

In [ ]:
df_jan["hour"] = df_jan.datahora_transacao.dt.hour
 
 
def share_diff(col: str) -> pd.DataFrame:
    a = df[col].value_counts(normalize=True).rename("train_2025")
    b = df_jan[col].value_counts(normalize=True).rename("jan_2026")
    out = pd.concat([a, b], axis=1).fillna(0)
    out["abs_diff"] = (out.train_2025 - out.jan_2026).abs()
    return out.round(4)
 
 
for c in ["tipo_estabelecimento", "cidade", "device_type", "is_new_device", "turno"]:
    print(f"\n--- {c} ---")
    print(share_diff(c).to_string())

Drift check on numeric variables

In [ ]:
q = [0.1, 0.5, 0.9, 0.99]
num = pd.concat(
    {
        "train_2025": df["valor_transacao"].quantile(q),
        "jan_2026": df_jan["valor_transacao"].quantile(q),
    },
    axis=1,
)
print("valor_transacao quantiles:")
print(num.round(2).to_string())

## 02.7 Which variables carry information?

Mutual information ranking

In [ ]:
feats = pd.DataFrame({
    "hour": df.hour,
    "is_new_device": df.is_new_device.astype(int),
    "amount_bin": pd.qcut(df.valor_transacao, 20, duplicates="drop").cat.codes,
    "tipo_estabelecimento": df.tipo_estabelecimento,
    "nome_estabelecimento": df.nome_estabelecimento,
    "cidade": df.cidade,
    "device_type": df.device_type,
    "dayofweek": df.dayofweek,
    "month": df.month.astype(str),
    "score_interno_bin": pd.qcut(df.score_interno, 20, duplicates="drop").cat.codes,
    "temperatura_bin": pd.qcut(df.temperatura_celsius, 20, duplicates="drop").cat.codes,
})
y = df[TARGET].astype(int)
mi = pd.Series({c: mutual_info_score(feats[c], y) / np.log(2) for c in feats})  # bits
print("Mutual information with the target (bits), higher = more informative:")
print(mi.sort_values(ascending=False).round(5).to_string())


Shallow decision tree to discover rules and interactions

In [ ]:
X = pd.get_dummies(
    pd.DataFrame({
        "hour": df.hour,
        "is_new_device": df.is_new_device.astype(int),
        "amount": df.valor_transacao,
        "tipo": df.tipo_estabelecimento,
        "cidade": df.cidade,
        "device": df.device_type,
    }),
    columns=["tipo", "cidade", "device"],
)
tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200, class_weight="balanced", random_state=42)
tree.fit(X, y)
print("Note: the weights are class-balanced, so the valuable infos are splits and the predicted class, not the raw counts.\n")
print(export_text(tree, feature_names=list(X.columns), show_weights=True))

## 02.8 Simple rule based benchmark (and the temporal split)
A lookup table of fraud rates, fit on the first 70% of the period and evaluated on the last 30%.
Any machine learning model should beat this benchmark to be worth its complexity.

Temporal split and lookup benchmark

In [ ]:
cut = df.datahora_transacao.quantile(0.7)
train, test = df[df.datahora_transacao <= cut], df[df.datahora_transacao > cut]
print(f"Train: {len(train):,} rows up to {cut} | fraud rate {train[TARGET].mean():.2%}")
print(f"Test : {len(test):,} rows after that | fraud rate {test[TARGET].mean():.2%}")
print(f"Random-score PR-AUC would be about {test[TARGET].mean():.4f}\n")
 
 
def lookup_score(keys: list[str], m: float = 20.0) -> np.ndarray:
    """Smoothed fraud rate of each key combination, estimated on train."""
    g = train.groupby(keys)[TARGET].agg(["sum", "count"])
    g["rate"] = (g["sum"] + m * train[TARGET].mean()) / (g["count"] + m)
    return test.join(g["rate"], on=keys)["rate"].fillna(train[TARGET].mean()).values
 
 
key_sets = [
    ["hour"],
    ["is_new_device"],
    ["hour", "is_new_device"],
    ["hour", "is_new_device", "tipo_estabelecimento"],
    ["hour", "is_new_device", "tipo_estabelecimento", "amount_bin"],
]
res = []
for ks in key_sets:
    s = lookup_score(ks)
    res.append({"keys": " + ".join(ks), "PR-AUC": average_precision_score(test[TARGET], s),
                "ROC-AUC": roc_auc_score(test[TARGET], s)})
print(pd.DataFrame(res).round(4).to_string(index=False))

Gain table of the best lookup

In [ ]:
best = key_sets[-1]
test = test.assign(score=lookup_score(best))
test = test.sort_values("score", ascending=False).reset_index(drop=True)
test["decile"] = pd.qcut(test.index, 10, labels=False) + 1
gain = test.groupby("decile")[TARGET].agg(n="size", frauds="sum")
gain["share_of_all_frauds"] = gain.frauds / gain.frauds.sum()
gain["cumulative_share"] = gain.share_of_all_frauds.cumsum()
print("Reviewing the top X% of transactions by risk captures this share of all frauds:")
print(gain.round(3).to_string())

## 02.9 Financial profile of fraud

Money

In [ ]:
total = df.valor_transacao.sum()
fraud_value = df.loc[df[TARGET], "valor_transacao"].sum()
print(f"Total value: R$ {total:,.0f} | fraud value: R$ {fraud_value:,.0f} ({fraud_value / total:.2%} of value)")
print(f"Fraud is {df[TARGET].mean():.2%} of transactions but {fraud_value / total:.2%} of value.")
print(f"Average ticket: fraud R$ {df.loc[df[TARGET], 'valor_transacao'].mean():.2f} | "
      f"legitimate R$ {df.loc[~df[TARGET], 'valor_transacao'].mean():.2f}")
print("\nFraud value by month:")
print(df[df[TARGET]].groupby("month")["valor_transacao"].sum().round(0).to_string())
print("\nFraud value by merchant type:")
print(df[df[TARGET]].groupby("tipo_estabelecimento")["valor_transacao"].agg(["count", "sum", "mean"]).round(1).to_string())

## 02.10 Customer history features (past info only)
Everything here uses only transactions that happened *before* the current one, so the same features can be built for January 2026 without leakage.

### 02.10.1 Build historical features

In [ ]:
g = df.groupby("nome_cliente")
df["prior_tx"] = g.cumcount()
df["hours_since_prev"] = g["datahora_transacao"].diff().dt.total_seconds() / 3600
cum_value = g["valor_transacao"].cumsum() - df.valor_transacao
df["amount_vs_past_mean"] = df.valor_transacao / (cum_value / df.prior_tx.replace(0, np.nan))
df["new_city_for_customer"] = (~df.duplicated(["nome_cliente", "cidade"])) & (df.prior_tx > 0)
df["new_merchant_for_customer"] = (~df.duplicated(["nome_cliente", "nome_estabelecimento"])) & (df.prior_tx > 0)
df["prior_frauds"] = g[TARGET].cumsum() - df[TARGET].astype(int)  # uses labels: see note below
 
print("Customers:", df.nome_cliente.nunique(), "| transactions per customer:")
print(df.groupby("nome_cliente").size().describe().round(2).to_string())

### 02.10.2 Fraud rate by historical feature

In [ ]:
def rate_by(series: pd.Series, name: str) -> None:
    t = df.groupby(series, observed=True)[TARGET].agg(n="size", fraud_rate="mean")
    print(f"\n--- {name} ---")
    print(t.round(4).to_string())
 
 
rate_by(pd.cut(df.prior_tx, [-1, 0, 1, 3, 7, np.inf], labels=["0", "1", "2-3", "4-7", "8+"]), "prior transactions of the customer")
rate_by(pd.cut(df.hours_since_prev, [0, 1, 6, 24, 24 * 7, 24 * 30, np.inf]), "hours since previous transaction")
rate_by(pd.cut(df.amount_vs_past_mean, [0, 0.25, 0.5, 1, 2, 4, np.inf]), "amount / customer past mean")
rate_by(df.new_city_for_customer, "city never used before by the customer")
rate_by(df.new_merchant_for_customer, "merchant never used before by the customer")
rate_by(df.prior_frauds.clip(upper=2), "frauds already seen on the customer (label history)")
print("\nNote: 'prior_frauds' depends on labels being known when scoring. In production there is a")
print("delay between a transaction and its confirmation as fraud, so validate this feature with care.")

### 02.10.3 Compare `is_new_device` with customer history

In [ ]:
print("is_new_device rate on the customer's FIRST transaction vs later ones:")
print(df.groupby(df.prior_tx == 0)["is_new_device"].mean().rename({True: "first tx", False: "later tx"}).round(4).to_string())

## 02.11 Fraud bursts at the same merchant

#### Burst test
Frauds are concentrated in few hours of the day, so raw gaps between frauds look "short", even without any burst. The fair test compare, for each merchant-day-hour cell, the observed numer of frauds with the number expected from hour, new device and merchant type alone.

In [ ]:
keys = ["hour", "is_new_device", "tipo_estabelecimento"]
tbl = df.groupby(keys)[TARGET].agg(["sum", "count"])
tbl["p_hat"] = (tbl["sum"] + 20 * p) / (tbl["count"] + 20)
df["p_hat"] = df.join(tbl["p_hat"], on=keys)["p_hat"].values
df["var_hat"] = df.p_hat * (1 - df.p_hat)
 
cells = df.groupby(["nome_estabelecimento", "date", "hour"]).agg(
    obs=(TARGET, "sum"), exp=("p_hat", "sum"), var=("var_hat", "sum"))
cells["z"] = (cells.obs - cells.exp) / np.sqrt(cells["var"].clip(lower=1e-9))
disp = ((cells.obs - cells.exp) ** 2).groupby(level=0).sum() / cells["var"].groupby(level=0).sum()
 
print("Dispersion index per merchant (about 1 = no bursts, clearly above 1 = frauds cluster in time):")
print(disp.round(3).to_string())
print("\nMost unusual merchant-hour cells:")
print(cells.reindex(cells.z.abs().sort_values(ascending=False).index).head(10).round(2).to_string())
print(f"\nCells analysed: {len(cells):,} | cells with |z| > 3 expected by chance: ~{len(cells) * 0.0027:.0f}"
      f" | observed: {int((cells.z.abs() > 3).sum())}")


Merchant fraud rate by month (compromised merchant overtime?)

In [ ]:
print(df.pivot_table(index="nome_estabelecimento", columns="month", values=TARGET, aggfunc="mean").round(3).to_string())

# 03. Feature Engineering

To be completed after the full-year EDA findings are confirmed. Keep only features available at transaction time and document the treatment of any customer history.


# 04. Model Development and Temporal Validation

To be completed: compare the simple rule benchmark with a linear baseline and a tree-based model. Report precision, recall, F1, PR-AUC, and the chosen business threshold on a chronological holdout.


# 05. January 2026 Scoring and Deliverables

To be completed: refit the selected approach on the appropriate 2025 data, score January 2026, export the classified CSV, and summarize the results and limitations. January 2026 has no ground-truth labels, so its predictive performance cannot be measured directly.
